<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/CalibratedTypedDecisionDatasetBuilder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Calibrated-Typed-Decisions (v1) - Dataset Builder**


---



---



Builds an open dataset of typed decisions (choice / yes-no) where every row has:
  - the human gold label (from a public labelled dataset)
  - the teacher's raw option log-probs (one forward pass, logit-slicing)
  - a calibrated soft label (teacher log-probs / T, with T fit by log loss
    against gold labels on a validation split)

## [1] Install (once per Colab session)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 46.1 MB/s eta 0:00:00


## [2] Google Drive (survives disconnects) + Hugging Face login (needs a WRITE token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import notebook_login
notebook_login()

## [3] Config

In [ ]:
import os, json, random
import numpy as np
import torch

SMOKE = True          # True: tiny end-to-end test run. False: the real build.

SEED        = 0
HF_REPO     = "m-lagnajit/calibrated-typed-decisions"
TEACHER     = "Qwen/Qwen2.5-7B-Instruct"
OUT_DIR     = "/content/drive/MyDrive/ctd_smoke" if SMOKE else "/content/drive/MyDrive/ctd"
N_PER_SPLIT = ({"train": 16, "validation": 16, "test": 16} if SMOKE
               else {"train": 1000, "validation": 200, "test": 200})
NOUL_RATE        = 0.3     # fraction of rows turned into yes/no questions
MIN_OPTIONS      = 3
MAX_OPTIONS      = 8
BATCH_SIZE       = 8
MAX_STATE_CHARS  = 1200
MIN_LETTER_MASS  = 0.5     # drop rows where the teacher didn't really answer with a letter
DATASET_LICENSE  = "cc-by-4.0"   # set only after checking it's compatible with EVERY kept source
os.makedirs(OUT_DIR, exist_ok=True)

SOURCES = [
    dict(name="banking77", path="PolyAI/banking77", config=None,
         text="text", label="label", train="train", eval="test",
         instructions="Which banking intent best describes this customer message?",
         noul="Is the customer's intent in this message: {label}?"),
    dict(name="clinc_oos", path="clinc/clinc_oos", config="plus",
         text="text", label="intent", train="train", eval="test",
         instructions="Which intent best describes this request to a virtual assistant?",
         noul="Is this request asking for: {label}?"),
    dict(name="go_emotions", path="google-research-datasets/go_emotions", config="simplified",
         text="text", label="labels", multi=True, train="train", eval="test",
         instructions="Which emotion does this comment express?",
         noul="Does this comment express {label}?"),
    dict(name="ag_news", path="fancyzhx/ag_news", config=None,
         text="text", label="label", train="train", eval="test",
         instructions="Which topic is this news article about?",
         noul="Is this news article about {label}?"),
    dict(name="emotion", path="dair-ai/emotion", config=None,
         text="text", label="label", train="train", eval="test",
         instructions="Which emotion does this text express?",
         noul="Does this text express {label}?"),
]

# Only these licenses pass the gate automatically. NC (non-commercial) and SA (share-alike)
# licenses, and anything "unknown"/"other", are excluded unless you read the card yourself.
PERMISSIVE = {"cc-by-4.0", "cc-by-3.0", "cc-by-2.0", "apache-2.0", "mit", "cc0-1.0"}
LICENSE_OVERRIDES = {}   # e.g. {"some/dataset": "cc-by-4.0"}  -- only after reading its card

## [4] License gate

In [ ]:
from huggingface_hub import dataset_info

def hub_license(path):
    try:
        cd = dataset_info(path).card_data
        lic = getattr(cd, "license", None) if cd is not None else None
    except Exception as e:
        return f"error:{type(e).__name__}"
    if isinstance(lic, list):
        lic = ",".join(lic)
    return (lic or "none").lower()

kept = []
print(f"{'source':42s} {'license':20s} kept?")
for src in SOURCES:
    lic = LICENSE_OVERRIDES.get(src["path"], hub_license(src["path"]))
    src["license"] = lic
    ok = lic in PERMISSIVE
    print(f"{src['path']:42s} {lic:20s} {'YES' if ok else 'no - check card manually'}")
    if ok:
        kept.append(src)
SOURCES = kept

source                                     license              kept?
PolyAI/banking77                           cc-by-4.0            YES
clinc/clinc_oos                            cc-by-3.0            YES
google-research-datasets/go_emotions       apache-2.0           YES
fancyzhx/ag_news                           unknown              no - check card manually
dair-ai/emotion                            other                no - check card manually


## [5] Load sources and build the remixed questions (no model needed yet)

In [ ]:
from datasets import load_dataset

def humanize(s):
    return s.replace("_", " ").strip()

def load_source(src):
    tr = load_dataset(src["path"], src["config"], split=src["train"])
    ev = load_dataset(src["path"], src["config"], split=src["eval"])
    feat = tr.features[src["label"]]
    names = feat.feature.names if src.get("multi") else feat.names
    return tr, ev, [humanize(n) for n in names]

def pick(ds, src, n, rng):
    """Pick n usable examples in a seeded random order."""
    order = list(range(len(ds)))
    rng.shuffle(order)
    out = []
    for i in order:
        ex = ds[i]
        lab = ex[src["label"]]
        if src.get("multi"):
            if len(lab) != 1:          # keep only single-label examples
                continue
            lab = lab[0]
        text = (ex[src["text"]] or "").strip()
        if not text:
            continue
        out.append((i, text[:MAX_STATE_CHARS], int(lab)))
        if len(out) == n:
            break
    return out

def make_question(src, names, gold, rng):
    """Turn one labelled example into a choice or yes/no question.
    display      = option texts in the order the teacher sees them
    display_perm = display position d shows canonical option display_perm[d]"""
    if rng.random() < NOUL_RATE:
        is_true = rng.random() < 0.5
        cand = gold if is_true else rng.choice([j for j in range(len(names)) if j != gold])
        if rng.random() < 0.5:
            display, perm = ["no", "yes"], [0, 1]     # canonical slots: [false, true]
        else:
            display, perm = ["yes", "no"], [1, 0]
        return dict(type="noul",
                    instructions=src["noul"].format(label=names[cand]),
                    options=["false", "true"], gold_index=int(is_true),
                    display=display, display_perm=perm)

    k = max(2, min(len(names), rng.randint(MIN_OPTIONS, MAX_OPTIONS)))
    others = rng.sample([j for j in range(len(names)) if j != gold], k - 1)
    opts = [gold] + others
    rng.shuffle(opts)                     # the right answer lands in a random slot
    options = [names[j] for j in opts]
    return dict(type="choice", instructions=src["instructions"],
                options=options, gold_index=opts.index(gold),
                display=options, display_perm=list(range(k)))

ALL = {"train": [], "validation": [], "test": []}
LOADED = []
for src in SOURCES:
    try:
        tr, ev, names = load_source(src)
    except Exception as e:
        print(f"SKIP {src['name']}: {type(e).__name__}: {e}")
        continue
    LOADED.append(src)
    rng = random.Random(f"{SEED}-{src['name']}")     # deterministic per source
    nv, nt = N_PER_SPLIT["validation"], N_PER_SPLIT["test"]
    picked = {"train": pick(tr, src, N_PER_SPLIT["train"], rng)}
    ev_rows = pick(ev, src, nv + nt, rng)            # val and test: disjoint halves of source test
    picked["validation"], picked["test"] = ev_rows[:nv], ev_rows[nv:]
    for split, items in picked.items():
        for (i, text, gold) in items:
            q = make_question(src, names, gold, rng)
            ALL[split].append(dict(
                id=f"{src['name']}-{split}-{i:06d}",
                source=src["path"], source_license=src["license"],
                state=text, gold=q["options"][q["gold_index"]], **q))
    print(f"{src['name']:12s} labels={len(names):3d}  " +
          "  ".join(f"{s}={len(v)}" for s, v in picked.items()))

print("rows:", {s: len(v) for s, v in ALL.items()})

README.md:   0%|          | 0.00/9.78k [00:00<?, ?B/s]

banking77.py:   0%|          | 0.00/7.17k [00:00<?, ?B/s]

SKIP banking77: RuntimeError: Dataset scripts are no longer supported, but found banking77.py


README.md:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

plus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  312kB            

...

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

go_emotions  labels= 28  train=16  validation=16  test=16
rows: {'train': 32, 'validation': 32, 'test': 32}


## [6] Load the teacher in 4-bit (a 7B in fp16 needs ~15 GB, a T4 has ~15 GB total)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(TEACHER)
tok.padding_side = "left"        # so position -1 is the real last token for every row in a batch
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    TEACHER,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16),   # T4 has no bfloat16
    device_map="auto")
model.eval()

# Each answer letter can be tokenized with or without a leading space ("A" vs " A").
# After the chat template, the first answer token is usually "A" with no space.
# We collect both and combine them, instead of guessing one.
LETTER_IDS = []
for d in range(MAX_OPTIONS):
    L = chr(65 + d)
    ids = set()
    for s in (L, " " + L):
        enc = tok.encode(s, add_special_tokens=False)
        if len(enc) == 1:
            ids.add(enc[0])
    LETTER_IDS.append(sorted(ids))
print("letter token ids:", LETTER_IDS)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

letter token ids: [[32, 362], [33, 425], [34, 356], [35, 422], [36, 468], [37, 434], [38, 479], [39, 472]]


## [7] Score every row with ONE forward pass (batched, resumable)

In [ ]:
SYSTEM = "You are a careful text classifier. Reply with only the letter of the single best option."

def build_prompt(r):
    block = "\n".join(f"{chr(65 + d)}. {t}" for d, t in enumerate(r["display"]))
    user = (f"{r['instructions']}\n\nText:\n{r['state']}\n\n"
            f"Options:\n{block}\n\nReply with only the letter.")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score_batch(rows):
    enc = tok([build_prompt(r) for r in rows], return_tensors="pt", padding=True).to(model.device)
    try:
        out = model(**enc, logits_to_keep=1)   # only run the LM head at the last position
    except TypeError:
        out = model(**enc)                     # older transformers: head runs at every position
    logp = torch.log_softmax(out.logits[:, -1, :].float(), dim=-1)    # (B, vocab)
    results = []
    for b, r in enumerate(rows):
        n = len(r["display"])
        disp = torch.stack([torch.logsumexp(logp[b, LETTER_IDS[d]], dim=0)
                            for d in range(n)]).cpu()                 # display order
        canon = torch.empty_like(disp)
        canon[torch.tensor(r["display_perm"])] = disp                 # back to canonical order
        results.append(dict(
            teacher_option_logprobs=[round(x, 5) for x in canon.tolist()],
            teacher_letter_mass=round(disp.exp().sum().item(), 5)))
    return results

def run_split(split):
    path = f"{OUT_DIR}/{split}.jsonl"
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            done = {json.loads(line)["id"] for line in f}
    todo = [r for r in ALL[split] if r["id"] not in done]
    print(f"{split}: {len(done)} already done, {len(todo)} to score")
    with open(path, "a") as f:
        for s in range(0, len(todo), BATCH_SIZE):
            batch = todo[s:s + BATCH_SIZE]
            for r, extra in zip(batch, score_batch(batch)):
                f.write(json.dumps({**r, **extra}, ensure_ascii=False) + "\n")
            f.flush()
            if (s // BATCH_SIZE) % 25 == 0:
                print(f"  {split}: {len(done) + s + len(batch)}/{len(ALL[split])}")

# validation + test first: you can check calibration quality before spending time on train
for split in ["validation", "test", "train"]:
    run_split(split)

validation: 0 already done, 32 to score
  validation: 8/32
test: 0 already done, 32 to score
  test: 8/32
train: 0 already done, 32 to score
  train: 8/32


## [8] Fit one temperature per (source, type) on VALIDATION, report on TEST

In [ ]:
def load_split(split):
    with open(f"{OUT_DIR}/{split}.jsonl") as f:
        rows = [json.loads(line) for line in f]
    good = [r for r in rows if r["teacher_letter_mass"] >= MIN_LETTER_MASS]
    return good, len(rows) - len(good)

DATA, DROPPED = {}, {}
for split in ["train", "validation", "test"]:
    DATA[split], DROPPED[split] = load_split(split)
print("dropped (teacher did not answer with a letter):", DROPPED)

def probs_at(r, T):
    z = np.array(r["teacher_option_logprobs"]) / T
    z = z - z.max()
    p = np.exp(z)
    return p / p.sum()

def nll(rows, T):
    """Log loss: a strictly proper scoring rule. Lowest when probabilities match reality."""
    return float(np.mean([-np.log(probs_at(r, T)[r["gold_index"]] + 1e-12) for r in rows]))

def ece(rows, T, n_bins=10):
    conf = np.array([probs_at(r, T).max() for r in rows])
    corr = np.array([int(probs_at(r, T).argmax() == r["gold_index"]) for r in rows])
    total = 0.0
    for b in range(n_bins):
        lo, hi = b / n_bins, (b + 1) / n_bins
        m = (conf >= lo) & ((conf < hi) if b < n_bins - 1 else (conf <= hi))
        if m.any():
            total += m.mean() * abs(corr[m].mean() - conf[m].mean())
    return float(total), float(corr.mean())

GRID = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))
TEMPS, REPORT = {}, []
for g in sorted({(r["source"], r["type"]) for r in DATA["validation"]}):
    val  = [r for r in DATA["validation"] if (r["source"], r["type"]) == g]
    test = [r for r in DATA["test"]       if (r["source"], r["type"]) == g]
    if len(val) < 5 or not test:
        continue
    T = float(GRID[np.argmin([nll(val, t) for t in GRID])])
    TEMPS[g] = T
    e_raw, acc = ece(test, 1.0)
    e_cal, _   = ece(test, T)
    REPORT.append(dict(source=g[0], type=g[1], n_test=len(test), teacher_acc=acc, T=T,
                       ece_raw=e_raw, ece_cal=e_cal,
                       nll_raw=nll(test, 1.0), nll_cal=nll(test, T)))

print(f"\n{'source':40s} {'type':6s} {'n':>4s} {'acc':>6s} {'T':>6s} {'ECE raw':>8s} {'ECE cal':>8s}")
for x in REPORT:
    print(f"{x['source']:40s} {x['type']:6s} {x['n_test']:>4d} {x['teacher_acc']:>6.3f} "
          f"{x['T']:>6.2f} {x['ece_raw']:>8.3f} {x['ece_cal']:>8.3f}")

for split in DATA:
    for r in DATA[split]:
        T = TEMPS.get((r["source"], r["type"]), 1.0)
        r["teacher_temperature"] = round(T, 4)
        r["teacher_probs"] = [round(float(p), 6) for p in probs_at(r, T)]

dropped (teacher did not answer with a letter): {'train': 0, 'validation': 0, 'test': 0}

source                                   type      n    acc      T  ECE raw  ECE cal
clinc/clinc_oos                          choice   12  1.000   0.25    0.000    0.000
clinc/clinc_oos                          noul      4  0.750   0.25    0.250    0.250
google-research-datasets/go_emotions     choice   10  0.600  12.05    0.354    0.272
google-research-datasets/go_emotions     noul      6  1.000  20.00    0.020    0.344


## [9] LOOK AT RAW ROWS before publishing

In [ ]:
for r in random.Random(1).sample(DATA["test"], min(3, len(DATA["test"]))):
    show = {k: r[k] for k in ["source", "type", "instructions", "state", "options", "gold",
                              "teacher_probs", "teacher_temperature", "teacher_letter_mass"]}
    print(json.dumps(show, indent=2, ensure_ascii=False)[:1500])
    print("-" * 60)

{
  "source": "clinc/clinc_oos",
  "type": "noul",
  "instructions": "Is this request asking for: last maintenance?",
  "state": "oil type and do it yourself oil change information please",
  "options": [
    "false",
    "true"
  ],
  "gold": "false",
  "teacher_probs": [
    1.0,
    0.0
  ],
  "teacher_temperature": 0.25,
  "teacher_letter_mass": 0.99984
}
------------------------------------------------------------
{
  "source": "clinc/clinc_oos",
  "type": "choice",
  "instructions": "Which intent best describes this request to a virtual assistant?",
  "state": "please speak a bit faster",
  "options": [
    "are you a bot",
    "who do you work for",
    "change speed"
  ],
  "gold": "change speed",
  "teacher_probs": [
    0.0,
    0.0,
    1.0
  ],
  "teacher_temperature": 0.25,
  "teacher_letter_mass": 1.0
}
------------------------------------------------------------
{
  "source": "google-research-datasets/go_emotions",
  "type": "choice",
  "instructions": "Which emotion doe

## [10] Publish: PRIVATE first, review on the Hub, then flip to public

In [ ]:
from datasets import Dataset, DatasetDict
from huggingface_hub import DatasetCard

KEEP = ["id", "source", "source_license", "type", "instructions", "state", "options",
        "gold", "gold_index", "display_perm",
        "teacher_option_logprobs", "teacher_temperature", "teacher_probs", "teacher_letter_mass"]

dd = DatasetDict({s: Dataset.from_list([{k: r[k] for k in KEEP} for r in rows])
                  for s, rows in DATA.items()})
print(dd)

table = "\n".join(
    f"| `{x['source']}` | {x['type']} | {x['n_test']} | {x['teacher_acc']:.3f} | "
    f"{x['T']:.2f} | {x['ece_raw']:.3f} | {x['ece_cal']:.3f} |" for x in REPORT)
source_list = "\n".join(f"- `{s['path']}`, license `{s['license']}`" for s in LOADED)

CARD = f"""
# calibrated-typed-decisions (v1)

Typed decisions (multiple choice and yes/no) over short English texts, each with a human
gold label **and** a calibrated soft label from a teacher LLM. Built for training and
evaluating small decision models whose confidence scores mean something.

## Why

A model that says 0.9 should be right about 90% of the time. Instruction-tuned LLMs read
as classifiers tend to be overconfident. This dataset pairs every decision with a teacher
probability distribution that has been calibrated against real answers, plus the raw
ingredients so you can redo the calibration your own way.

## Splits

| split | rows |
|---|---|
| train | {len(DATA['train'])} |
| validation | {len(DATA['validation'])} |
| test | {len(DATA['test'])} |

## Fields

| field | meaning |
|---|---|
| `type` | `choice` or `noul` (yes/no; options are always `["false", "true"]`) |
| `instructions` | the question |
| `state` | the text being judged |
| `options` | option labels in canonical order |
| `gold`, `gold_index` | the human label from the source dataset |
| `teacher_option_logprobs` | teacher log-probability of each option's answer letter, canonical order |
| `teacher_temperature` | temperature fit for this row's (source, type) group |
| `teacher_probs` | `softmax(teacher_option_logprobs / teacher_temperature)` |
| `teacher_letter_mass` | total probability the teacher put on answer letters at all |
| `display_perm` | the option order the teacher saw (display position to canonical index) |

## How the labels were made

1. Examples come from public, permissively licensed classification datasets (listed below).
2. Each example becomes a question with 3 to 8 shuffled options (the gold label plus random
   distractors), or a yes/no question about one candidate label (about 30% of rows).
3. The teacher (`{TEACHER}`, 4-bit NF4) reads each question once. The answer-letter
   log-probabilities at the first answer position are recorded. Nothing is generated.
4. Rows where the teacher put under {MIN_LETTER_MASS:.0%} of its probability on answer letters are dropped
   (train {DROPPED['train']}, validation {DROPPED['validation']}, test {DROPPED['test']}).
5. One temperature per (source, type) is fit on the validation split by minimizing log loss
   against gold labels, then applied to every row. Calibration is reported on the test split.

## Teacher calibration on the test split

| source | type | n | teacher acc | T | ECE raw | ECE calibrated |
|---|---|---|---|---|---|---|
{table}

## Sources

{source_list}

## Limitations

- Soft labels reflect one teacher's judgment and biases, not human disagreement.
- Calibration is fit per group on average; individual rows can still be off.
- Distractors are sampled at random, so many choice questions are easier than hard-negative ones.
- Options are humanized source label names without descriptions.
- English only. Gold labels inherit any noise in the source datasets.
- The teacher ran in 4-bit; full-precision logits would differ slightly.

## Usage

```python
from datasets import load_dataset
ds = load_dataset("{HF_REPO}")
row = ds["train"][0]
print(row["options"], row["teacher_probs"], row["gold"])
```
"""

if SMOKE:
    print("SMOKE run: not publishing. Set SMOKE = False and rerun from cell [3].")
else:
    dd.push_to_hub(HF_REPO, private=True)
    card = DatasetCard.load(HF_REPO)
    card.data.license = DATASET_LICENSE
    card.data.language = ["en"]
    card.data.task_categories = ["text-classification"]
    card.data.tags = ["calibration", "soft-labels", "distillation", "typed-decisions"]
    card.data.pretty_name = "Calibrated Typed Decisions"
    card.text = CARD
    card.push_to_hub(HF_REPO)
    print(f"Pushed (private): https://huggingface.co/datasets/{HF_REPO}")

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'source_license', 'type', 'instructions', 'state', 'options', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_probs', 'teacher_letter_mass'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['id', 'source', 'source_license', 'type', 'instructions', 'state', 'options', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_probs', 'teacher_letter_mass'],
        num_rows: 32
    })
    test: Dataset({
        features: ['id', 'source', 'source_license', 'type', 'instructions', 'state', 'options', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_probs', 'teacher_letter_mass'],
        num_rows: 32
    })
})
SMOKE run: not publishing. Set SMOKE = False and rerun from cell [3].
